In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()
telemetry = spark.read.table(f"{SILVER}.telemetry_clean")
dim_machine = spark.read.table(f"{GOLD}.dim_machine").filter(col("__CURRENT") == True)

# 1. FACT TELEMETRY (30-Min Tumbling Window)
# 👉 FIX: date_sk ab INT yyyyMMdd hai — dim_date.date_sk se seedha join hota hai
#    (pehle DATE tha, dim me INT tha — join kabhi match nahi hota tha)
telemetry_bucketed = telemetry.withColumn(
    "time_bucket_30m",
    expr("timestampadd(MINUTE, -minute(sensor_ts) % 30, date_trunc('minute', sensor_ts))")
)
fact_telemetry = telemetry_bucketed.groupBy("machine_id", "time_bucket_30m").agg(
    avg("engine_temp").alias("avg_temp"), max("engine_temp").alias("max_temp"),
    avg("hydraulic_press").alias("avg_press"), min("hydraulic_press").alias("min_press"),
    avg("fuel_level").alias("avg_fuel"), avg("rpm").alias("avg_rpm"), count("event_id").alias("readings")
).join(dim_machine.select("machine_sk", "machine_id"), "machine_id", "left") \
 .select("machine_sk", expr("date_format(time_bucket_30m, 'HHmm')").cast("int").alias("time_bucket_sk"), date_format("time_bucket_30m", "yyyyMMdd").cast("int").alias("date_sk"), "time_bucket_30m", "avg_temp", "max_temp", "avg_press", "min_press", "avg_fuel", "avg_rpm", "readings")
fact_telemetry.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.fact_telemetry_30min")

# 2. FACT FAULT EVENT (Event Level)
faults = spark.read.table(f"{SILVER}.telemetry_clean").filter(col("fault_code").isNotNull())
fact_fault = faults.join(dim_machine.select("machine_sk", "machine_id"), "machine_id", "left") \
    .select(expr("xxhash64(event_id)").alias("fault_sk"), "machine_sk", date_format("sensor_ts", "yyyyMMdd").cast("int").alias("date_sk"), col("sensor_ts").alias("fault_ts"), col("fault_code").alias("severity"))
fact_fault.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.fact_fault_event")
print("✅ gold_04_fact_telemetry built (date_sk INT yyyyMMdd).")